In [0]:
from pyspark.sql import functions as F

SILVER_DOCUMENTS = "dbr_dev_ua5816bd.gulidov_artem_silver.silver_documents"
SILVER_CHUNKS = "dbr_dev_ua5816bd.gulidov_artem_silver.document_chunks"

CHUNK_SIZE = 120
CHUNK_OVERLAP = 20
CHUNK_STEP = CHUNK_SIZE - CHUNK_OVERLAP

documents = spark.table(SILVER_DOCUMENTS)

chunks = documents.withColumn(
    "words",
    F.split(F.col("context"), r"\s+")
)

chunks = chunks.withColumn(
    "chunk_indexes",
    F.sequence(
        F.lit(0),
        F.ceil(
            F.size("words") / F.lit(CHUNK_STEP)
        ).cast("int") - 1
    )
)

chunks = chunks.withColumn(
    "chunk_index",
    F.explode(F.col("chunk_indexes"))
)

chunks = chunks.withColumn(
    "chunk_text",
    F.concat_ws(
        " ",
        F.slice(
            F.col("words"),
            F.col("chunk_index") * CHUNK_STEP + 1,
            CHUNK_SIZE
        )
    )
)

chunks = chunks.withColumn(
    "chunk_id",
    F.sha2(
        F.concat_ws(
            "_",
            F.col("document_id"),
            F.col("chunk_index").cast("string")
        ),
        256
    )
)

chunks = chunks.select(
    "chunk_id",
    "document_id",
    "title",
    "source",
    "chunk_index",
    "chunk_text"
)

(
    chunks.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(SILVER_CHUNKS)
)

print(f"Total chunks: {chunks.count():,}")